In [3]:
from datetime import datetime, timedelta
from typing import List, Optional
from enum import Enum

class EstadoMaterial(Enum):
    DISPONIBLE = "disponible"
    PRESTADO = "prestado"
    EN_TRANSFERENCIA = "en_transferencia"

class Material:
    def __init__(self, id: int, titulo: str):
        self.id = id
        self.titulo = titulo
        self.estado = EstadoMaterial.DISPONIBLE
        self.ubicacion_actual = None
        self.disponible = True

    def verificar_disponibilidad(self) -> bool:
        return self.estado == EstadoMaterial.DISPONIBLE

    def actualizar_estado(self, nuevo_estado: EstadoMaterial):
        self.estado = nuevo_estado
        self.disponible = (nuevo_estado == EstadoMaterial.DISPONIBLE)

class Libro(Material):
    def __init__(self, id: int, titulo: str, autor: str, genero: str, isbn: str):
        super().__init__(id, titulo)
        self.autor = autor
        self.genero = genero
        self.isbn = isbn

class Revista(Material):
    def __init__(self, id: int, titulo: str, edicion: str, periodicidad: str, numero: int):
        super().__init__(id, titulo)
        self.edicion = edicion
        self.periodicidad = periodicidad
        self.numero = numero

class MaterialDigital(Material):
    def __init__(self, id: int, titulo: str, tipo_archivo: str, enlace_descarga: str, tamano_mb: int):
        super().__init__(id, titulo)
        self.tipo_archivo = tipo_archivo
        self.enlace_descarga = enlace_descarga
        self.tamano_mb = tamano_mb

class Persona:
    def __init__(self, id: int, nombre: str, email: str, telefono: str):
        self.id = id
        self.nombre = nombre
        self.email = email
        self.telefono = telefono

class Usuario(Persona):
    def __init__(self, id: int, nombre: str, email: str, telefono: str):
        super().__init__(id, nombre, email, telefono)
        self.historial_prestamos = []
        self.penalizaciones = []
        self.multas_pendientes = 0.0

    def consultar_catalogo(self, catalogo: 'Catalogo', criterio: str) -> List[Material]:
        return catalogo.buscar_en_todas_sucursales(criterio)

    def ver_historial_prestamos(self) -> List['Prestamo']:
        return self.historial_prestamos

    def pagar_multa(self, monto: float) -> bool:
        if monto <= self.multas_pendientes:
            self.multas_pendientes -= monto
            return True
        return False

class Bibliotecario(Persona):
    def __init__(self, id: int, nombre: str, email: str, telefono: str, codigo_empleado: str):
        super().__init__(id, nombre, email, telefono)
        self.codigo_empleado = codigo_empleado
        self.sucursal_asignada = None

    def transferir_material(self, material: Material, sucursal_destino: 'Sucursal') -> bool:
        if material.ubicacion_actual == self.sucursal_asignada:
            material.actualizar_estado(EstadoMaterial.EN_TRANSFERENCIA)
            material.ubicacion_actual = sucursal_destino
            sucursal_destino.recibir_transferencia(material)
            return True
        return False

    def aplicar_penalizacion(self, usuario: Usuario, monto: float, motivo: str):
        penalizacion = Penalizacion(
            id=len(usuario.penalizaciones) + 1,
            usuario=usuario,
            monto=monto,
            motivo=motivo
        )
        usuario.penalizaciones.append(penalizacion)
        usuario.multas_pendientes += monto

class Sucursal:
    def __init__(self, id: int, nombre: str, direccion: str):
        self.id = id
        self.nombre = nombre
        self.direccion = direccion
        self.catalogo_local = []
        self.encargado = None

    def buscar_material(self, criterio: str) -> List[Material]:
        return [m for m in self.catalogo_local 
                if criterio.lower() in m.titulo.lower() or 
                (isinstance(m, Libro) and criterio.lower() in m.autor.lower())]

    def recibir_transferencia(self, material: Material):
        material.actualizar_estado(EstadoMaterial.DISPONIBLE)
        if material not in self.catalogo_local:
            self.catalogo_local.append(material)

class Catalogo:
    def __init__(self, sucursales: List[Sucursal]):
        self.sucursales = sucursales

    def buscar_en_todas_sucursales(self, criterio: str) -> dict:
        resultados = {}
        for sucursal in self.sucursales:
            materiales = sucursal.buscar_material(criterio)
            if materiales:
                resultados[sucursal.nombre] = materiales
        return resultados

class Prestamo:
    def __init__(self, id: int, usuario: Usuario, material: Material):
        self.id = id
        self.usuario = usuario
        self.material = material
        self.fecha_prestamo = datetime.now()
        self.fecha_devolucion_esperada = self.fecha_prestamo + timedelta(days=14)
        self.fecha_devolucion_real = None
        self.estado = "activo"

    def registrar_devolucion(self):
        self.fecha_devolucion_real = datetime.now()
        self.estado = "devuelto"
        self.material.actualizar_estado(EstadoMaterial.DISPONIBLE)
        self.verificar_retraso()

    def verificar_retraso(self) -> int:
        if self.fecha_devolucion_real:
            dias_retraso = (self.fecha_devolucion_real - self.fecha_devolucion_esperada).days
            return max(0, dias_retraso)
        return 0

class Penalizacion:
    def __init__(self, id: int, usuario: Usuario, monto: float, motivo: str):
        self.id = id
        self.usuario = usuario
        self.monto = monto
        self.motivo = motivo
        self.fecha = datetime.now()
        self.pagada = False

    def registrar_pago(self):
        self.pagada = True

# Ejemplo de uso que demuestra los retos
def ejemplo_biblioteca():
    # Crear sucursales
    sucursal_centro = Sucursal(1, "Biblioteca Central", "Centro")
    sucursal_norte = Sucursal(2, "Biblioteca Norte", "Norte")
    
    # Crear catálogo general
    catalogo = Catalogo([sucursal_centro, sucursal_norte])
    
    # Crear materiales
    libro1 = Libro(1, "1984", "George Orwell", "Ficción", "123-456")
    libro2 = Libro(2, "Cien años de soledad", "Gabriel García Márquez", "Ficción", "789-012")
    
    # Agregar materiales a sucursales
    libro1.ubicacion_actual = sucursal_centro
    libro2.ubicacion_actual = sucursal_norte
    sucursal_centro.catalogo_local.append(libro1)
    sucursal_norte.catalogo_local.append(libro2)
    
    # Crear personal y usuarios
    bibliotecario = Bibliotecario(1, "Ana López", "ana@biblioteca.com", "1234567", "EMP001")
    bibliotecario.sucursal_asignada = sucursal_centro
    usuario = Usuario(1, "Juan Pérez", "juan@email.com", "7654321")
    
    print("=== Demostración del sistema ===")
    
    # 1. Buscar materiales en todas las sucursales
    print("\n1. Búsqueda en todas las sucursales:")
    resultados = catalogo.buscar_en_todas_sucursales("ficción")
    for sucursal, materiales in resultados.items():
        print(f"En {sucursal}:")
        for material in materiales:
            print(f"- {material.titulo}")
    
    # 2. Transferir material entre sucursales
    print("\n2. Transferencia de material:")
    print(f"Ubicación inicial de '{libro1.titulo}': {libro1.ubicacion_actual.nombre}")
    bibliotecario.transferir_material(libro1, sucursal_norte)
    print(f"Nueva ubicación de '{libro1.titulo}': {libro1.ubicacion_actual.nombre}")
    
    # 3. Sistema de penalizaciones
    print("\n3. Sistema de penalizaciones:")
    # Simular un préstamo con retraso
    prestamo = Prestamo(1, usuario, libro2)
    prestamo.fecha_prestamo = datetime.now() - timedelta(days=20)  # Simular préstamo de hace 20 días
    prestamo.fecha_devolucion_esperada = prestamo.fecha_prestamo + timedelta(days=14)
    prestamo.registrar_devolucion()
    
    dias_retraso = prestamo.verificar_retraso()
    if dias_retraso > 0:
        multa = dias_retraso * 1.0  # $1 por día de retraso
        bibliotecario.aplicar_penalizacion(
            usuario,
            multa,
            f"Retraso de {dias_retraso} días en la devolución"
        )
        print(f"Multa aplicada: ${multa} por {dias_retraso} días de retraso")
        print(f"Total de multas pendientes: ${usuario.multas_pendientes}")

if __name__ == "__main__":
    ejemplo_biblioteca()

=== Demostración del sistema ===

1. Búsqueda en todas las sucursales:

2. Transferencia de material:
Ubicación inicial de '1984': Biblioteca Central
Nueva ubicación de '1984': Biblioteca Norte

3. Sistema de penalizaciones:
Multa aplicada: $6.0 por 6 días de retraso
Total de multas pendientes: $6.0


In [2]:
def ejemplo_completo_biblioteca():
    print("\n=== SISTEMA DE BIBLIOTECA DIGITAL ===\n")
    
    # 1. Crear sucursales
    print("1. Creando sucursales...")
    sucursal_centro = Sucursal(1, "Biblioteca Central", "Av. Principal 123")
    sucursal_norte = Sucursal(2, "Biblioteca Norte", "Calle Norte 456")
    sucursal_sur = Sucursal(3, "Biblioteca Sur", "Av. Sur 789")
    
    # 2. Crear catálogo general
    print("2. Inicializando catálogo general...")
    catalogo = Catalogo([sucursal_centro, sucursal_norte, sucursal_sur])
    
    # 3. Crear diferentes tipos de materiales
    print("3. Registrando materiales...")
    # Libros
    libros = [
        Libro(1, "1984", "George Orwell", "Ficción Distópica", "123-456"),
        Libro(2, "Cien años de soledad", "Gabriel García Márquez", "Realismo Mágico", "789-012"),
        Libro(3, "El Principito", "Antoine de Saint-Exupéry", "Literatura Infantil", "345-678")
    ]
    
    # Revistas
    revistas = [
        Revista(4, "National Geographic", "Enero 2024", "Mensual", 289),
        Revista(5, "Scientific American", "Febrero 2024", "Mensual", 156)
    ]
    
    # Materiales Digitales
    digitales = [
        MaterialDigital(6, "Curso Python", "PDF", "python_curso.pdf", 25),
        MaterialDigital(7, "Revista Digital", "EPUB", "revista_tech.epub", 15)
    ]
    
    # 4. Distribuir materiales en sucursales
    print("4. Distribuyendo materiales en sucursales...")
    # Central
    for material in [libros[0], revistas[0], digitales[0]]:
        material.ubicacion_actual = sucursal_centro
        sucursal_centro.catalogo_local.append(material)
    
    # Norte
    for material in [libros[1], revistas[1]]:
        material.ubicacion_actual = sucursal_norte
        sucursal_norte.catalogo_local.append(material)
    
    # Sur
    for material in [libros[2], digitales[1]]:
        material.ubicacion_actual = sucursal_sur
        sucursal_sur.catalogo_local.append(material)
    
    # 5. Crear personal
    print("5. Registrando personal...")
    bibliotecarios = [
        Bibliotecario(1, "Ana López", "ana@biblioteca.com", "1234567", "EMP001"),
        Bibliotecario(2, "Carlos Ruiz", "carlos@biblioteca.com", "2345678", "EMP002"),
        Bibliotecario(3, "María González", "maria@biblioteca.com", "3456789", "EMP003")
    ]
    
    # Asignar bibliotecarios a sucursales
    bibliotecarios[0].sucursal_asignada = sucursal_centro
    bibliotecarios[1].sucursal_asignada = sucursal_norte
    bibliotecarios[2].sucursal_asignada = sucursal_sur
    
    # 6. Crear usuarios
    print("6. Registrando usuarios...")
    usuarios = [
        Usuario(1, "Juan Pérez", "juan@email.com", "7654321"),
        Usuario(2, "Laura Sánchez", "laura@email.com", "8765432"),
        Usuario(3, "Pedro Ramírez", "pedro@email.com", "9876543")
    ]
    
    print("\n=== OPERACIONES DEL SISTEMA ===\n")
    
    # 7. Realizar búsquedas
    print("7. Búsquedas en el catálogo:")
    print("\nBúsqueda por género 'Ficción':")
    resultados = catalogo.buscar_en_todas_sucursales("Ficción")
    for sucursal, materiales in resultados.items():
        print(f"\nEn {sucursal}:")
        for material in materiales:
            print(f"- {material.titulo}")
    
    # 8. Realizar préstamos
    print("\n8. Gestión de préstamos:")
    # Préstamo normal
    prestamo1 = Prestamo(1, usuarios[0], libros[0])
    usuarios[0].historial_prestamos.append(prestamo1)
    libros[0].actualizar_estado(EstadoMaterial.PRESTADO)
    print(f"Préstamo realizado: {libros[0].titulo} a {usuarios[0].nombre}")
    
    # Préstamo con retraso (simulado)
    prestamo2 = Prestamo(2, usuarios[1], libros[1])
    prestamo2.fecha_prestamo = datetime.now() - timedelta(days=20)
    prestamo2.fecha_devolucion_esperada = prestamo2.fecha_prestamo + timedelta(days=14)
    usuarios[1].historial_prestamos.append(prestamo2)
    print(f"Préstamo realizado (simulado con retraso): {libros[1].titulo} a {usuarios[1].nombre}")
    
    # 9. Procesar devoluciones y penalizaciones
    print("\n9. Procesando devoluciones y penalizaciones:")
    # Devolución normal
    prestamo1.registrar_devolucion()
    print(f"Devolución normal: {libros[0].titulo}")
    
    # Devolución con retraso
    prestamo2.registrar_devolucion()
    dias_retraso = prestamo2.verificar_retraso()
    if dias_retraso > 0:
        multa = dias_retraso * 1.0  # $1 por día
        bibliotecarios[0].aplicar_penalizacion(
            usuarios[1],
            multa,
            f"Retraso de {dias_retraso} días en la devolución"
        )
        print(f"Devolución con retraso: {libros[1].titulo}")
        print(f"Penalización aplicada: ${multa} por {dias_retraso} días de retraso")
    
    # 10. Transferencias entre sucursales
    print("\n10. Transferencias entre sucursales:")
    print(f"Ubicación inicial de '{libros[2].titulo}': {libros[2].ubicacion_actual.nombre}")
    bibliotecarios[2].transferir_material(libros[2], sucursal_centro)
    print(f"Nueva ubicación de '{libros[2].titulo}': {libros[2].ubicacion_actual.nombre}")
    
    # 11. Consultar historial de usuario
    print("\n11. Historial de usuario:")
    for prestamo in usuarios[1].ver_historial_prestamos():
        print(f"- {prestamo.material.titulo}: {'Devuelto' if prestamo.fecha_devolucion_real else 'En préstamo'}")
    
    # 12. Gestión de multas
    print("\n12. Estado de multas:")
    for usuario in usuarios:
        if usuario.multas_pendientes > 0:
            print(f"{usuario.nombre}: ${usuario.multas_pendientes} pendientes")
            # Simular pago parcial
            monto_pago = usuario.multas_pendientes / 2
            if usuario.pagar_multa(monto_pago):
                print(f"Pago parcial realizado: ${monto_pago}")
                print(f"Multa restante: ${usuario.multas_pendientes}")
    
    # 13. Verificar disponibilidad de materiales
    print("\n13. Estado actual de materiales:")
    for material in libros + revistas + digitales:
        print(f"- {material.titulo}: {material.estado.value}")
        print(f"  Ubicación: {material.ubicacion_actual.nombre}")

if __name__ == "__main__":
    ejemplo_completo_biblioteca()


=== SISTEMA DE BIBLIOTECA DIGITAL ===

1. Creando sucursales...
2. Inicializando catálogo general...
3. Registrando materiales...
4. Distribuyendo materiales en sucursales...
5. Registrando personal...
6. Registrando usuarios...

=== OPERACIONES DEL SISTEMA ===

7. Búsquedas en el catálogo:

Búsqueda por género 'Ficción':

8. Gestión de préstamos:
Préstamo realizado: 1984 a Juan Pérez
Préstamo realizado (simulado con retraso): Cien años de soledad a Laura Sánchez

9. Procesando devoluciones y penalizaciones:
Devolución normal: 1984
Devolución con retraso: Cien años de soledad
Penalización aplicada: $6.0 por 6 días de retraso

10. Transferencias entre sucursales:
Ubicación inicial de 'El Principito': Biblioteca Sur
Nueva ubicación de 'El Principito': Biblioteca Central

11. Historial de usuario:
- Cien años de soledad: Devuelto

12. Estado de multas:
Laura Sánchez: $6.0 pendientes
Pago parcial realizado: $3.0
Multa restante: $3.0

13. Estado actual de materiales:
- 1984: disponible
  U